In [6]:
# --- STEP 1: SETUP ---
# Install the necessary libraries
! pip install python-dotenv --upgrade --quiet langchain-groq

import os
import getpass
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Securely set your Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")



In [7]:
# --- STEP 2: DEFINE EXPERTS ---
# We use Mixtral 8x7B for high capacity and specialized personas
MODEL_CONFIG = {
    "technical": "You are a Senior Technical Support Engineer. Provide rigorous, code-focused, and precise troubleshooting steps. Avoid fluff.",
    "billing": "You are a Billing Specialist. Be empathetic and focus on financial policies and refund procedures. Keep it professional.",
    "general": "You are a friendly Customer Success Manager. Handle general inquiries with a warm and helpful tone."
}



In [8]:
# --- UPDATED STEP 3: THE ROUTER ---
def route_prompt(user_input):
    # Using the 'instant' model for the router because it's fast and cheap
    router_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)
    router_prompt = ChatPromptTemplate.from_template(
        "Classify the intent of this customer query into exactly one of these categories: "
        "[technical, billing, general]. Return ONLY the single word name of the category.\n\n"
        "Query: {query}"
    )
    chain = router_prompt | router_llm | StrOutputParser()
    return chain.invoke({"query": user_input}).lower().strip()



In [9]:
# --- UPDATED STEP 4: THE ORCHESTRATOR ---
def process_request(user_input):
    category = route_prompt(user_input)
    print(f"--- Routing to: {category.upper()} EXPERT ---")

    system_prompt = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])

    # Using the 'versatile' 70B model for the actual experts to get high-quality answers
    expert_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
    expert_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{query}")
    ])

    chain = expert_prompt | expert_llm | StrOutputParser()
    return chain.invoke({"query": user_input})

In [10]:
# --- TEST IT ---
print(process_request("My python script is throwing an IndexError on line 5."))
# print(process_request("I was charged twice for my subscription this month."))

--- Routing to: TECHNICAL EXPERT ---
To troubleshoot the `IndexError`, we'll need to examine the code surrounding line 5. 

1. **Verify the Line Number**: Ensure the error message is accurately pointing to line 5 in your script.
2. **Check Index Access**: On line 5, identify any array, list, or string indexing operations. Look for syntax like `my_list[index]` or `my_string[index]`.
3. **Validate Index Value**: Confirm that the `index` variable or value is within the valid range for the data structure being accessed. For example, if `my_list` has 5 elements, valid indices are 0 through 4.
4. **Review Data Structure Initialization**: Before line 5, ensure the data structure (list, array, string) is properly initialized and populated with the expected data.
5. **Print Debugging Statements**: Add print statements before line 5 to verify the data structure's length and the index value being used. For example:
    ```python
print("Data Structure Length:", len(my_list))
print("Index Value:", 